In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 112.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import xgboost as xgb
import shap
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import sklearn
import math

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")

In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelf from Python file in the drive
!cp /content/drive/MyDrive/...../woodelf.py /content/

import woodelf

# PDP Code

In [ ]:
class CPDVMetric(woodelf.CubeMetric):
    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}
        pdp_values = {}
        if len(s_plus) == 1:
            for f in s_plus:
                pdp_values[f] = 1
        if len(s_plus) == 0:
            for f in s_minus:
                pdp_values[f] = -1
        return pdp_values

In [ ]:
class PathToValuesMatrixLimitSPlus(woodelf.PathToValuesMatrix):
    # Ignored all cubes with |S^+| > MAX_S_PLUS_SIZE to reduce the complexity element of TL3**D to TL2**D*(D**MAX_S_PLUS_SIZE)
    MAX_S_PLUS_SIZE = NotImplemented

    @classmethod
    def map_patterns_to_cube(cls, features_in_path: List[str]):
        updated_wdnf_table = {0: {0: (set(), set())}}
        current_wdnf_table = None
        for feature in features_in_path:
            current_wdnf_table = updated_wdnf_table
            updated_wdnf_table = {}
            for consumer_pattern in current_wdnf_table:
                updated_wdnf_table[consumer_pattern * 2 + 0] = {}
                updated_wdnf_table[consumer_pattern * 2 + 1] = {}
                for background_pattern in current_wdnf_table[consumer_pattern]:
                    s_plus, s_minus = current_wdnf_table[consumer_pattern][background_pattern]

                    # The implementation is identical to the PathToValuesMatrix.map_patterns_to_cube implementation, except for this if.
                    if len(s_plus | {feature}) <= cls.MAX_S_PLUS_SIZE:
                        updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 0] = (s_plus | {feature}, s_minus) # Rule 1

                    updated_wdnf_table[consumer_pattern * 2 + 0][background_pattern * 2 + 1] = (s_plus, s_minus | {feature}) # Rule 2
                    updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 1] = (s_plus, s_minus) # Rule 3

        return updated_wdnf_table

class PathToValuesMatrixLimitSPlusTo1(PathToValuesMatrixLimitSPlus):
    # We uses the fact CPDVMetric ignored all cubes with |S^+| > 1 to reduce the complexity element of TL3**D to TL2**D*D
    MAX_S_PLUS_SIZE = 1

class PathToValuesMatrixLimitSPlusTo2(PathToValuesMatrixLimitSPlus):
    # We uses the fact PDIVOrder1Or2 ignored all cubes with |S^+| > 2 to reduce the complexity element of TL3**D to TL(2**D)*(D**2)
    MAX_S_PLUS_SIZE = 2

In [ ]:
def build_sampled_points_df(data: pd.DataFrame, k: int, seed: int = None):
    """
    Sample k points from every column.
    """
    sample_points_data = {}
    for f in data.columns:
        sample_points_data[f] = list(data[f].sample(k, random_state=seed))
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_equally_distanced_points_df(data: pd.DataFrame, k: int, percentiles: Tuple[float]):
    """
    Take equally distanced points from each column. The min point will be in the precentile percentiles[0]
    and the max point will be in the precentile percentiles[1].
    This is also the default implementation of sklearn
    """
    sample_points_data = {}
    for f in data.columns:
        low, high = np.percentile(data[f].dropna(), [percentiles[0] * 100, percentiles[1]*100])
        # get k equally spaced points between them
        points = np.linspace(low, high, k)
        sample_points_data[f] = list(points)
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_points_for_full_pdp(data: pd.DataFrame, model, as_df: bool=True):
    """
    Provide the points that will create a full PDP - a graph the will provide the PDV for every x value.
    Does this by collecting all the threshold values from the model. See Sect. of the paper.
    """
    # load the model
    model_objs = woodelf.load_decision_tree_ensamble_model(model, list(data.columns))

    # collect all the theshold values for each feature
    th_values = {f: [] for f in list(data.columns)}
    for tree in model_objs:
        for node in tree.bfs(including_myself=True, including_leaves=False):
            th_values[node.feature_name].append(node.value)

    # Make sure the thesholds are unique and sort them
    for f in th_values:
        th_values[f] = sorted(list(set(th_values[f])))

    if not as_df:
        return th_values

    # zfill
    max_th_length = max([len(thersholds) for thersholds in th_values.values()])
    for f in th_values:
        th_values[f].extend([0] * (max_th_length - len(th_values[f])) )

    # from the built thershold build the points Data Frame
    return pd.DataFrame(th_values)

In [ ]:
def build_points_for_pdp(model, data: pd.DataFrame, k: int = 100, percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose : bool = True):
    start_time = time.time()
    if sampled:
        points_df = build_sampled_points_df(data, k, seed)
    elif full_pdp:
        points_df = build_points_for_full_pdp(data, model)
    else:
        points_df = build_equally_distanced_points_df(data, k, percentiles)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec")
    return points_df

def woodelf_pdp(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=True)
    return woodelf_pdp_given_points_df(model, data, points_df, accurate, centered, GPU), points_df

def woodelf_pdp_given_points_df(model, data: pd.DataFrame, sampled_points_df: pd.DataFrame, accurate: bool = True, centered: bool = True, GPU: bool = False):
    """
    Compute all the PDVs of the provided points. Use WOODELF!
    """
    metric=CPDVMetric()
    p2v = PathToValuesMatrixLimitSPlusTo1(metric)
    if accurate:
        pdvs = woodelf.calculate_background_metric(model, consumer_data=sampled_points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    else:
        pdvs = woodelf.calculate_path_dependent_metric(model, consumer_data=sampled_points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    if centered:
        return pdvs

    avg_prediction = float(model.predict(data).mean())
    for f in pdvs:
        pdvs[f] += avg_prediction
    return pdvs

## Joint DPD code

In [ ]:
# PDP joint

from itertools import combinations

def all_subsets_of_size_0_1_2(s):
    subsets = [set()]
    for k in [1,2]:
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIVOrder1Or2(woodelf.CubeMetric):
    INTERACTION_VALUE = True

    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets_of_size_0_1_2(s_minus):
            s = tuple(s_plus | sm)
            if len(s) in [1,2]:
                pdivs[s] = (-1) ** (len(sm))
        return pdivs

def bits(n, D):
    bs = []
    for i in range(D):
        bs.append(n % 2)
        n = n // 2
    return reversed(bs)

def build_points_for_joint_pdp(points_df: pd.DataFrame):
    D = math.ceil(math.log2(len(points_df.columns)))
    data = {f: [] for f in points_df.columns}
    k = len(points_df)
    for i, f in enumerate(points_df.columns):
        for b in bits(i, D):
            if b == 0:
                data[f].extend(np.tile(points_df[f].values, k))
            elif b == 1:
                data[f].extend(np.repeat(points_df[f].values, k))
    return pd.DataFrame(data)

def first_different_bit(n1, n2, D):
    assert n1 != n2
    i = 0
    for b1, b2 in zip(bits(n1, D), bits(n2, D)):
        if b1 != b2:
            return i
        i += 1

def clip_result(pdvs, features, k):
    D = math.ceil(math.log2(len(features)))
    feature_to_index = {f:i for i,f in enumerate(features)}
    clipped = {}
    for f1, f2 in pdvs:
        i1 = feature_to_index[f1]
        i2 = feature_to_index[f2]
        h = first_different_bit(i1, i2, D)
        clipped[(f1, f2)] = pdvs[(f1, f2)][h*(k**2): (h+1)*(k**2)]
    return clipped


def woodelf_pdp_joint(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose: bool = True):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    start_time = time.time()
    original_points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=False)
    if full_pdp:
        k = len(original_points_df)

    points_df = build_points_for_joint_pdp(original_points_df)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec. The size of the created df {len(points_df)}")

    metric = PDIVOrder1Or2()
    p2v = PathToValuesMatrixLimitSPlusTo2(metric)
    if accurate:
        pdivs = woodelf.calculate_background_metric(
            model, consumer_data=points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    else:
        pdivs = woodelf.calculate_path_dependent_metric(
            model, consumer_data=points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    avg_prediction = float(model.predict(data).mean())
    base_pdv = np.array([avg_prediction] * len(points_df))
    zero_array = np.array([0] * len(points_df))
    pdvs = {}

    D = math.ceil(math.log2(len(points_df.columns)))
    points_parts = {f: [points_df[f].values[i:i + k**2] for i in range(0, len(points_df[f]), k**2)] for f in data.columns}
    f1_points = {}
    f2_points = {}
    for i, f1 in enumerate(data.columns):
        for j, f2 in enumerate(data.columns):
            if f1 != f2:
                pair = (f1, f2)
                pdvs[(f1,f2)] = base_pdv + pdivs.get((f1,), zero_array) + pdivs.get((f2,), zero_array) + pdivs.get(pair, zero_array)
                points_part_index = first_different_bit(i,j,D)
                f1_points[(f1,f2)] = points_parts[f1][points_part_index]
                f2_points[(f1,f2)] = points_parts[f2][points_part_index]
    clipped_pdvs = clip_result(pdvs, list(data.columns), k)
    return clipped_pdvs, f1_points, f2_points

## Any Order PDIV code

In [ ]:
from itertools import combinations

def all_subsets(s):
    subsets = []
    for k in range(len(s) + 1):
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIV(woodelf.CubeMetric):
    INTERACTION_VALUE = True
    def calc_metric(
        self, s_plus: Set, s_minus: Set
    ) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets(s_minus):
            s = tuple(s_plus | sm)
            pdivs[s] = (-1) ** (len(sm))
        return pdivs

# Fraud Data + Model

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=6,
    max_leaf_nodes=None,
    # tree_method="hist",  # use 'gpu_hist' if you want GPU support
    random_state=42
)
gradient_boosting_model.fit(fraud_train, transactions_train['isFraud'])

y_pred = gradient_boosting_model.predict(transactions_test[train_features])
print(f"Accuracy: {accuracy_score(transactions_test['isFraud'], y_pred.round())}, F1 score: {f1_score(transactions_test['isFraud'], y_pred.round())}")

Accuracy: 0.9656246824939886, F1 score: 0.365625


## Woodelf PDP computation



In [ ]:
def get_pdp_testing_params():
    return {
        "Exact PDP k=5":      dict(k = 5,         accurate = True,  centered = False, GPU = False),
        "Exact PDP k=10":     dict(k = 10,        accurate = True,  centered = False, GPU = False),
        "Exact PDP k=100":    dict(k = 100,       accurate = True,  centered = False, GPU = False),
        "Exact full PDP":     dict(full_pdp=True, accurate = True,  centered = False, GPU = False),
        "Estimated PDP k=5":  dict(k = 5,         accurate = False, centered = True,  GPU = False),
    }

def messure_woodelf_pdp_times(model, data, params):
    running_results = {}
    for name, running_params in params.items():
        print(name + ":")
        start_time = time.time()
        woodelf_pdp(model, data, **running_params)
        running_results[name] = time.time() - start_time
        print()

    full_pdp_df_length = build_points_for_full_pdp(data, model).shape[0]
    print(f"The legnth of the consumer data in the full PDP run is {full_pdp_df_length}")

    print("running times:")
    for name in running_results:
        print(f"{name} took: {running_results[name]} sec")

In [ ]:
messure_woodelf_pdp_times(gradient_boosting_model, data=fraud_train, params=get_pdp_testing_params())

Exact PDP k=5:
Building the points took: 3.3509738445281982 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:08<00:00, 11.86it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 138.58it/s]



Exact PDP k=10:
Building the points took: 3.399078607559204 sec


...


Estimated PDP k=5:
Building the points took: 3.4579920768737793 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 245.30it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 154.76it/s]



The legnth of the consumer data in the full PDP run is 66
running times:
Exact PDP k=5 took: 14.501279354095459 sec
Exact PDP k=10 took: 15.230731725692749 sec
Exact PDP k=100 took: 14.798039436340332 sec
Exact full PDP took: 11.801071405410767 sec
Estimated PDP k=5 took: 4.622673988342285 sec


In [ ]:
start_time = time.time()
woodelf_pdp_joint(gradient_boosting_model, fraud_train, k = 5, accurate = True, centered=False, GPU = False)
print(f"\n Exact Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 3.4952006340026855 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [00:09<00:00, 10.85it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 67.04it/s]



 Exact Joint PDP k=5 computation took: 18.977829933166504 sec


In [ ]:
start_time = time.time()
woodelf_pdp_joint(gradient_boosting_model, fraud_train, k = 5, accurate = False, centered=False, GPU = False)
print(f"\n Estimated Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 3.3971755504608154 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 98.56it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 88.68it/s]



 Estimated Joint PDP k=5 computation took: 10.190206289291382 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    gradient_boosting_model, consumer_data=fraud_train.head(10_000), background_data=fraud_train, metric=PDIV(), global_importance=False, GPU=False
)
print(f"\n Any Order PDIVs n=10,000 took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:10<00:00,  9.17it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:09<00:00, 10.11it/s]


 Any Order PDIVs n=10,000 took: 20.95412015914917 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    gradient_boosting_model, consumer_data=fraud_train, background_data=fraud_train, metric=PDIV(), global_importance=True, GPU=False
)
print(f"\n Any Order PDIVs took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [00:10<00:00,  9.44it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [04:23<00:00,  2.64s/it]


 Any Order PDIVs took: 274.6564733982086 sec


## SOTA pdp computation - up to 3 hours running time...

This section can take up to 3 hours. It takes long as we didn't want to estimated running times.

In [ ]:
start_time = time.time()
failed_columns = []
# We have to run this one feature at a time, trying to provide significant more than 1 (say 16 features at a time) crashes the RAM :(
for f in tqdm(fraud_train.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=fraud_train, features=[f], feature_names=[f], grid_resolution=5, method='brute', kind='average'
        )
    except ValueError:
        print(f"{f} has failed...")
        failed_columns.append(f)

print(f"Exact PDP k=5 using sklean took: {time.time() - start_time} sec") # Took 2404 sec (40 min)

  3%|▎         | 11/397 [01:50<1:04:42, 10.06s/it]

C3 has failed...


 38%|███▊      | 149/397 [23:18<40:08,  9.71s/it]

V104 has failed...


 43%|████▎     | 169/397 [23:44<04:09,  1.10s/it]

...

V311 has failed...


 92%|█████████▏| 364/397 [54:20<05:09,  9.37s/it]

V319 has failed...


100%|██████████| 397/397 [58:30<00:00,  8.84s/it]

Exact PDP k=5 using sklean took: 3510.614492416382 sec


In [ ]:
start_time = time.time()
failed_columns = []
# We have to run this one feature at a time, trying to provide significant more than 1 (say 16 features at a time) crashes the RAM :(
for f in tqdm(fraud_train.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=fraud_train, features=[f], feature_names=[f], grid_resolution=10, method='brute', kind='average'
        )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Exact PDP k=10 using sklean took: {time.time() - start_time} sec") # Took 2404 sec (40 min)

100%|██████████| 397/397 [1:52:21<00:00, 16.98s/it]


15 Failed columns are: ['C3', 'V104', 'V111', 'V112', 'V113', 'V123', 'V125', 'V135', 'V281', 'V286', 'V297', 'V300', 'V301', 'V311', 'V319']
Exact PDP k=10 using sklean took: 6741.725870847702 sec


In [ ]:
points_for_full_pdp = build_points_for_full_pdp(data=fraud_train, model=gradient_boosting_model, as_df=False)
start_time = time.time()
failed_columns = []
for f in tqdm(fraud_train.columns):
    try:
        if len(points_for_full_pdp[f]) > 0:
            accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                estimator=gradient_boosting_model, X=fraud_train, features=[f], feature_names=[f], custom_values={f: points_for_full_pdp[f]}, method='brute', kind='average'
            )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Exact full PDP using sklean took: {time.time() - start_time} sec") # Took 1669 sec (28 min)

100%|██████████| 397/397 [59:17<00:00,  8.96s/it]


0 Failed columns are: []
Exact full PDP using sklean took: 3557.324166536331 sec


In [ ]:
start_time = time.time()
failed_columns = []
for f in tqdm(fraud_train.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=fraud_train, features=[f], feature_names=[f], grid_resolution=5, method='recursion', kind='average'
        )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Estimation of all 5 PDVs using sklean took: {time.time() - start_time} sec") # Took 2.366 sec

100%|██████████| 397/397 [00:03<00:00, 121.35it/s]


27 Failed columns are: ['C3', 'V104', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120', 'V121', 'V122', 'V123', 'V125', 'V135', 'V281', 'V286', 'V297', 'V300', 'V301', 'V311', 'V319']
Estimation of all 5 PDVs using sklean took: 3.2750837802886963 sec


### Joint PDP

In [ ]:
start_time = time.time()
features = fraud_train.columns
failures = 0
successes = 0
for i1 in tqdm(range(len(features))):
    for i2 in range(i1 + 1, len(features)):
        try:
            accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                estimator=gradient_boosting_model, X=fraud_train, features=[(i1,i2)], grid_resolution=5, method='recursion', kind='average'
            )
            successes += 1
        except ValueError:
            failures += 1

print()
print(f"Had {failures} failures and {successes} successful computations")
print(f"Estimated Joint PDP k=5 using sklean took: {time.time() - start_time} sec")

100%|██████████| 397/397 [19:26<00:00,  2.94s/it]


Had 10341 failures and 68265 successful computations
Estimated Joint PDP k=5 using sklean took: 1166.0814344882965 sec


In [ ]:
import random
pairs_indexes_10 = [(random.choice(range(0, len(fraud_train.columns))), random.choice(range(0, len(fraud_train.columns)))) for _ in range(10)]
pairs_indexes_10

[(98, 2),
 (364, 185),
 (187, 139),
 (372, 161),
 (67, 253),
 (4, 146),
 (363, 12),
 (280, 278),
 (307, 369),
 (215, 20)]

In [ ]:
start_time = time.time()
for fs in tqdm(pairs_indexes_10):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=fraud_train, features=[fs], grid_resolution=5, method='brute', kind='average'
        )
    except ValueError:
        print(f"{fs} has failed...")

joint_pdp_running_time = time.time() - start_time
print()
print(f"Exact Joint PDP of 10 pairs and k=5 using sklean took: {time.time() - start_time} sec") # Took 2.366 sec

 10%|█         | 1/10 [00:49<07:28, 49.89s/it]

(364, 185) has failed...


 30%|███       | 3/10 [01:29<03:12, 27.49s/it]

(372, 161) has failed...


100%|██████████| 10/10 [06:22<00:00, 38.23s/it]


Exact Joint PDP of 10 pairs and k=5 using sklean took: 382.26338601112366 sec


In [ ]:
number_of_pairs = (len(fraud_train.columns) * (len(fraud_train.columns) - 1)) / 2
running_time_per_single_pair = joint_pdp_running_time / 8 # 2 of 10 pairs have failed
print(f"Estimated computation time of 'Exact Joint PDP' on all pairs is: {(number_of_pairs * running_time_per_single_pair) / (60*60*24)} days")

Estimated computation time of 'Exact Joint PDP' on all pairs is: 34.777989686653015 days


# Empirical Correctness Verification

In [ ]:
TOLERANCE = 0.00001

In [ ]:
# Estimated PDP empirical correctness verification

trainset_1000_head = fraud_train.head(1000)

estimated_pdp_woodelf, sampled_points_estimated_woodelf = woodelf_pdp(
    gradient_boosting_model, data=trainset_1000_head, k = 5, accurate = False, centered = True, GPU = False, seed = 42
)

for f in fraud_train.columns:
    try:
        estimated_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=trainset_1000_head, features=[f], feature_names=[f], custom_values={f: sampled_points_estimated_woodelf[f]},
            method='recursion', kind='average'
        )
    except ValueError:
        print(f"{f} sklearn has failed...")

    if f not in estimated_pdp_woodelf:
        if np.max(np.abs(estimated_pdp_result_sklean['average'][0])) > TOLERANCE:
            print(f"{f} pdv should be all 0 but they are not")
    else:

        if np.max(np.abs(estimated_pdp_result_sklean['average'][0] - estimated_pdp_woodelf[f])) > TOLERANCE:
            print(f"{f} pdv do not agree")


Building the points took: 0.13158488273620605 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 261.66it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 157.09it/s]


In [ ]:
# Exact PDP empirical correctness verification

filledna_trainset_1000_head = fraud_train.head(1000)
avg_prediction = float(gradient_boosting_model.predict(filledna_trainset_1000_head).mean())

estimated_pdp_woodelf, sampled_points_estimated_woodelf = woodelf_pdp(
    gradient_boosting_model, data=filledna_trainset_1000_head, k = 5, accurate = True, centered = False, GPU = False, seed = 42
)

for f in tqdm(fraud_train.columns):
    try:
        estimated_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=filledna_trainset_1000_head, features=[f], feature_names=[f],
            custom_values={f: sampled_points_estimated_woodelf[f]}, method='brute', kind='average'
        )
    except ValueError:
        print(f"{f} sklearn has failed...")

    if f not in estimated_pdp_woodelf:
        if np.max(np.abs(estimated_pdp_result_sklean['average'][0] - avg_prediction)) > TOLERANCE:
            print(f"{f} pdv should be all the mean prediction but they are not")
    else:

        if np.max(np.abs(estimated_pdp_result_sklean['average'][0] - estimated_pdp_woodelf[f])) > TOLERANCE:
            print(f"{f} pdv do not agree")
            print(estimated_pdp_result_sklean['average'][0])
            print(estimated_pdp_woodelf[f])

Building the points took: 0.12571096420288086 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 83.82it/s]


cache misses: 49, cache used: 4406


100%|██████████| 397/397 [00:18<00:00, 21.47it/s]


In [ ]:
trainset_1000_head = fraud_train.head(1000)

estimated_pdp_woodelf, f1_points, f2_points = woodelf_pdp_joint(
    gradient_boosting_model, data=trainset_1000_head, k = 5, accurate = True, centered = False, GPU = False, seed = 42
)

for f1, f2 in pairs_indexes_10:
    feature_1 = list(fraud_train.columns)[f1]
    feature_2 = list(fraud_train.columns)[f2]


    for i, (v1, v2) in enumerate(zip(f1_points[(feature_1, feature_2)], f2_points[(feature_1, feature_2)])):
        current_train = trainset_1000_head.copy()
        current_train[feature_1] = v1
        current_train[feature_2] = v2
        direct_computation = gradient_boosting_model.predict(current_train).mean()
        if (feature_1, feature_2) in estimated_pdp_woodelf:
            woodelf_computation = estimated_pdp_woodelf[(feature_1, feature_2)][i]
        else:
            woodelf_computation = 0

        if abs(direct_computation - woodelf_computation) > TOLERANCE:
            print(f"{(feature_1, feature_2)} ({f1, f2}) pdv do not agree: {direct_computation} != {woodelf_computation}  ({v1=}, {v2=})")

Building the points took: 0.21753644943237305 sec. The size of the created df 225


Preprocessing the trees: 100%|██████████| 100/100 [00:01<00:00, 66.06it/s]


cache misses: 49, cache used: 4406


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 68.12it/s]


# Is the estimation method accurate?

In [ ]:
# Assuming the estimation method is centered while the brute is not

trainset_1000_head = fraud_train.head(1000)
avg_prediction = float(gradient_boosting_model.predict(trainset_1000_head).mean())

sampled_points_1000_head = build_equally_distanced_points_df(data=trainset_1000_head, k=5, percentiles=(0.05, 0.95))

number_of_disagrements = 0
for f in fraud_train.columns:
    try:
        acurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=trainset_1000_head, features=[f], feature_names=[f], custom_values={f: sampled_points_1000_head[f]}, method='brute', kind='average'
        )
        estimated_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=trainset_1000_head, features=[f], feature_names=[f], custom_values={f: sampled_points_1000_head[f]}, method='recursion', kind='average'
        )
    except ValueError:
        print(f"{f} sklearn has failed...")

    methods_diffs = np.abs(acurate_pdp_result_sklean['average'][0] - (estimated_pdp_result_sklean['average'][0] + avg_prediction))
    if np.max(methods_diffs) > 0.00001:
        number_of_disagrements += 1
        print(f"{number_of_disagrements}. {f} pdv do not agree. {np.sum(methods_diffs > 0.00001)} of the values have difference of above 0.00001 and the max diff is {np.max(methods_diffs)}")

print(f"total number of disagrements: {number_of_disagrements}")

1. TransactionAmt pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.00511092914975575
2. card1 pdv do not agree. 4 of the values have difference of above 0.00001 and the max diff is 0.0005057011316996173
3. card2 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.0002467773405053185
4. card3 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.00431355817303361
5. card5 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.00023079988279055622
6. addr1 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.0003647784846088978
7. addr2 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.0003377626925453181
8. dist1 pdv do not agree. 5 of the values have difference of above 0.00001 and the max diff is 0.00265622126889236
9. dist2 pdv do not agree. 5 of the values have diff

In [ ]:
# Assuming both the estimation method and the brute method are not centered

trainset_1000_head = fraud_train.head(1000)
avg_prediction = float(gradient_boosting_model.predict(trainset_1000_head).mean())

sampled_points_1000_head = build_equally_distanced_points_df(data=trainset_1000_head, k=5, percentiles=(0.05, 0.95))

number_of_disagrements = 0
for f in fraud_train.columns:
    try:
        acurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=trainset_1000_head, features=[f], feature_names=[f], custom_values={f: sampled_points_1000_head[f]}, method='brute', kind='average'
        )
        estimated_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=gradient_boosting_model, X=trainset_1000_head, features=[f], feature_names=[f], custom_values={f: sampled_points_1000_head[f]}, method='recursion', kind='average'
        )
    except ValueError:
        print(f"{f} sklearn has failed...")

    methods_diffs = np.abs(acurate_pdp_result_sklean['average'][0] - (estimated_pdp_result_sklean['average'][0] + 0)) # 0 and not avg_prediction
    if np.max(methods_diffs) > 0.0001:
        number_of_disagrements += 1
        print(f"{number_of_disagrements}. {f} pdv do not agree. {np.sum(methods_diffs > 0.0001)} of the values have difference of above 0.0001 and the max diff is {np.max(methods_diffs)}")

print(f"total number of disagrements: {number_of_disagrements}")

1. TransactionAmt pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.03127651445031137
2. card1 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.02667128643225524
3. card2 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.02641236264106094
4. card3 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.03047914347358923
5. card5 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.026396385183346176
6. addr1 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.02602477437995698
7. addr2 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.025827822608010302
8. dist1 pdv do not agree. 5 of the values have difference of above 0.0001 and the max diff is 0.02882180656944798
9. dist2 pdv do not agree. 5 of the values have difference of above 0

# KDD-Cup 1999: Intrusion Detection Dataset

In [ ]:
detection_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
unlabeled_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
small_test_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")

detection_train_features_names = [f for f in detection_data.columns if f != "target"]
detection_trainset = detection_data[detection_train_features_names]

In [ ]:
detection_model = sklearn.ensemble.HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=6,
    max_leaf_nodes=None,
    # tree_method="hist",  # use 'gpu_hist' if you want GPU support
    random_state=42
)
detection_model.fit(detection_trainset, detection_data['target'])

y_pred = detection_model.predict(small_test_data[detection_train_features_names])
print(f"Accuracy: {accuracy_score(small_test_data['target'], y_pred.round())}, F1 score: {f1_score(small_test_data['target'], y_pred.round())}")

Accuracy: 0.9264859546858975, F1 score: 0.9522245415206658


## Woodelf PDP computation


In [ ]:
messure_woodelf_pdp_times(detection_model, data=detection_trainset, params=get_pdp_testing_params())

Exact PDP k=5:
Building the points took: 5.635435104370117 sec


Preprocessing the trees: 100%|██████████| 100/100 [01:14<00:00,  1.34it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 170.15it/s]



Exact PDP k=10:
Building the points took: 5.608211278915405 sec


...


Estimated PDP k=5:
Building the points took: 5.5615034103393555 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 310.96it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 207.76it/s]



The legnth of the consumer data in the full PDP run is 44
running times:
Exact PDP k=5 took: 95.55039167404175 sec
Exact PDP k=10 took: 95.90197777748108 sec
Exact PDP k=100 took: 96.22089958190918 sec
Exact full PDP took: 90.34701442718506 sec
Estimated PDP k=5 took: 6.45103645324707 sec


In [ ]:
start_time = time.time()
joint_pdvs = woodelf_pdp_joint(detection_model, detection_trainset, k = 5, accurate = True, centered=False, GPU = False)
print(f"\n Exact Joint PDP k=5 computation took: {time.time() - start_time} sec")

Building the points took: 5.5376505851745605 sec. The size of the created df 175


Preprocessing the trees: 100%|██████████| 100/100 [01:14<00:00,  1.35it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:01<00:00, 98.99it/s]



 Exact Joint PDP k=5 computation took: 95.19871473312378 sec


In [ ]:
start_time = time.time()
joint_pdvs = woodelf_pdp_joint(detection_model, detection_trainset, k = 5, accurate = False, centered=False, GPU = False)
print(f"\n Estimated Joint PDP k=5 computation took: {time.time() - start_time} sec") # Took 21.5680 sec (didn't save due to google colab long run saving bug)

In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    detection_model, consumer_data=detection_trainset.head(10_000), background_data=detection_trainset, metric=PDIV(), global_importance=False, GPU=False
)
print(f"Any Order PDIVs n=10,000 took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [01:14<00:00,  1.34it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [00:05<00:00, 16.91it/s]

Any Order PDIVs n=10,000 took: 80.55946016311646 sec


In [ ]:
start_time = time.time()
woodelf.calculate_background_metric(
    detection_model, consumer_data=detection_trainset, background_data=detection_trainset, metric=PDIV(), global_importance=True, GPU=False
)
print(f"Any Order PDIVs took: {time.time() - start_time} sec")

Preprocessing the trees: 100%|██████████| 100/100 [01:14<00:00,  1.34it/s]


cache misses: 74, cache used: 3484


Computing the values: 100%|██████████| 100/100 [33:21<00:00, 20.01s/it]

Any Order PDIVs took: 2075.551076889038 sec


## SOTA pdp computation - up to 5 hours running time...

This section can take up to 5 hours. It takes long as we didn't want to estimated running times.

In [ ]:
start_time = time.time()
failed_columns = []
# We need to run it one by one, as several features together crashes the RAM
for f in tqdm(detection_trainset.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=detection_model, X=detection_trainset, features=[f], feature_names=[f], grid_resolution=5, method='brute', kind='average'
        )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Exact PDP k=5 using sklean took: {time.time() - start_time} sec")  # Took 4105 sec

100%|██████████| 121/121 [1:12:04<00:00, 35.74s/it]


8 Failed columns are: ['duration', 'urgent', 'hot', 'num_failed_logins', 'num_compromised', 'num_root', 'num_file_creations', 'num_access_files']
Exact PDP k=5 using sklean took: 4324.905223608017 sec


In [ ]:
start_time = time.time()
failed_columns = []
# We need to run it one by one, as several features together crashes the RAM
for f in tqdm(detection_trainset.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=detection_model, X=detection_trainset, features=[f], feature_names=[f], grid_resolution=10, method='brute', kind='average'
        )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Exact PDP k=10 using sklean took: {time.time() - start_time} sec")  # Took 4105 sec

100%|██████████| 121/121 [1:40:13<00:00, 49.70s/it]


6 Failed columns are: ['duration', 'hot', 'num_compromised', 'num_root', 'num_file_creations', 'num_access_files']
Exact PDP k=10 using sklean took: 6013.115252494812 sec


In [ ]:
points_for_full_pdp = build_points_for_full_pdp(data=detection_trainset, model=detection_model, as_df=False)
print(f"Full PDP require {sum([len(points) for points in points_for_full_pdp.values()])} points")
start_time = time.time()
failed_columns = []
for f in tqdm(detection_trainset.columns):
    try:
        if len(points_for_full_pdp[f]) > 0:
            accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                estimator=detection_model, X=detection_trainset, features=[f], feature_names=[f], custom_values={f: points_for_full_pdp[f]}, method='brute', kind='average'
            )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Exact full PDP using sklean took: {time.time() - start_time} sec") # Took 6007 sec

Full PDP require 483 points


100%|██████████| 121/121 [1:56:31<00:00, 57.78s/it]


0 Failed columns are: []
Exact full PDP using sklean took: 6991.212193250656 sec


In [ ]:
start_time = time.time()
failed_columns.append(f)
for f in tqdm(detection_trainset.columns):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=detection_model, X=detection_trainset, features=[f], feature_names=[f], grid_resolution=5, method='recursion', kind='average'
        )
    except ValueError:
        failed_columns.append(f)

print()
print(f"{len(failed_columns)} Failed columns are: {failed_columns}")
print(f"Estimated PDP k=5 using sklean took: {time.time() - start_time} sec") # Took 5.8 sec

100%|██████████| 121/121 [00:05<00:00, 22.16it/s]


9 Failed columns are: ['flag_S3', 'duration', 'urgent', 'hot', 'num_failed_logins', 'num_compromised', 'num_root', 'num_file_creations', 'num_access_files']
Estimated PDP k=5 using sklean took: 5.462949991226196 sec


### Joint PDP

In [ ]:
start_time = time.time()
features = detection_trainset.columns
failures = 0
successes = 0
for i1 in tqdm(range(len(features))):
    for i2 in range(i1 + 1, len(features)):
        try:
            accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                estimator=detection_model, X=detection_trainset, features=[(i1,i2)], grid_resolution=5, method='recursion', kind='average'
            )
            successes += 1
        except ValueError:
            failures += 1

print()
print(f"Had {failures} failures and {successes} successful computations")
print(f"Estimated Joint PDP k=5 using sklean took: {time.time() - start_time} sec") # Took 2.366 sec

100%|██████████| 121/121 [10:05<00:00,  5.01s/it]


Had 932 failures and 6328 successful computations
Estimated Joint PDP k=5 using sklean took: 605.6596629619598 sec


In [ ]:
import random
pairs_indexes_10 =[(random.choice(range(0, len(detection_trainset.columns))), random.choice(range(0, len(detection_trainset.columns)))) for _ in range(10)]
pairs_indexes_10

[(78, 21),
 (75, 102),
 (113, 82),
 (34, 1),
 (98, 84),
 (86, 59),
 (46, 33),
 (16, 30),
 (75, 65),
 (13, 13)]

In [ ]:
start_time = time.time()
for fs in tqdm(pairs_indexes_10):
    try:
        accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
            estimator=detection_model, X=detection_trainset, features=[fs], grid_resolution=5, method='brute', kind='average'
        )
    except ValueError:
        print(f"{fs} has failed...")

joint_pdp_running_time = time.time() - start_time
print()
print(f"Exact Joint PDP of 10 pairs and k=5 using sklean took: {joint_pdp_running_time} sec") # Took 2.366 sec

100%|██████████| 10/10 [17:10<00:00, 103.07s/it]

(13, 13) has failed...

Exact Joint PDP of 10 pairs and k=5 using sklean took: 1030.6884224414825 sec


In [ ]:
number_of_pairs = (len(detection_trainset.columns) * (len(detection_trainset.columns) - 1)) / 2
running_time_per_single_pair = joint_pdp_running_time / 10
print(f"Estimated computation time of 'Exact Joint PDP' on all pairs is: {(number_of_pairs * running_time_per_single_pair) / (60*60*24)} days")

Estimated computation time of 'Exact Joint PDP' on all pairs is: 8.660645771904123 days


# PDIVs estimation

### On fraud data

In [ ]:
loaded_model = woodelf.load_decision_tree_ensamble_model(gradient_boosting_model, train_features)
total_exponential_time = 0
n_features_lst = []
for tree in loaded_model:
    tree_features = [n.feature_name for n in tree.bfs(including_leaves=False)]
    # print(len(tree_features))
    n_features = len(set(tree_features))
    n_features_lst.append(n_features)
    total_exponential_time += 2 ** n_features
total_exponential_time
# n_features_lst

2199096386519040

In [ ]:
fastpd_running_time = 23*60 #seconds
fastpd_dataset_size = 311029
fastpd_number_of_features = 9
number_of_trees = 100

In [ ]:
c = (fastpd_running_time/((2**fastpd_number_of_features)*fastpd_dataset_size*number_of_trees))

In [ ]:
c

8.66579161428677e-08

In [ ]:
# estimated time for all data, in years
fraud_data_size= 472432
seconds_in_a_year = 60*60*24*365
(c * total_exponential_time * fraud_data_size) / seconds_in_a_year

2854862.566434205

In [ ]:
# estimated time for n=10,000, in years
(c * total_exponential_time * 10000) / seconds_in_a_year

60429.06844655326

## On KDD cup data

In [ ]:
loaded_model = woodelf.load_decision_tree_ensamble_model(detection_model, detection_train_features_names)
total_exponential_time = 0
n_features_lst = []
for tree in loaded_model:
    tree_features = [n.feature_name for n in tree.bfs(including_leaves=False)]
    n_features = len(set(tree_features))
    n_features_lst.append(n_features)
    total_exponential_time += 2 ** n_features
total_exponential_time

1490305024

In [ ]:
# estimated time for all data, in years
detection_data_size= 4898431
(c * total_exponential_time * detection_data_size) / seconds_in_a_year

20.06013238805841

In [ ]:
# estimated time for n=10,000, in days
(c * total_exponential_time * 10000) / (60*60*24)

14.947537939477598